# Batch CV Parser

Process a directory of PDF/DOCX CVs with parallel LLM augmentation. Outputs combined CSV or JSON.

In [61]:
from pathlib import Path

# Project root: parent of this notebook's directory
PROJECT_PATH = Path.cwd().parent if (Path.cwd() / "batch_cv_parser.ipynb").exists() else Path.cwd()

# --- User configuration (paths relative to PROJECT_PATH) ---
CV_INPUT_DIR = PROJECT_PATH / "cv"                # Directory containing PDF/DOCX files
TEMP_DIR = PROJECT_PATH / "tmp"                   # Metadata JSON and augmented artifacts
OUTPUT_FILE = PROJECT_PATH / "output" / "combined.csv"  # Final output path
OUTPUT_FORMAT = "csv"                             # "csv" or "json"
MAX_WORKERS = 2                                   # Parallel LLM calls (2-4 recommended)
OPENAI_MODEL = "gpt-5.2"                           # None = use OPENAI_MODEL from .env
MAX_COMPLETION_TOKENS = 96384                     # Model max (e.g. 16384 for gpt-4o-mini)
DEDUPLICATE_ROWS = True                          # When True, merge duplicate title+year rows (keep published over in_progress)

In [62]:
import os
from dotenv import load_dotenv
load_dotenv(PROJECT_PATH / ".env")
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not set. Create a .env file with OPENAI_API_KEY=your-key")
print("API key loaded from .env")

API key loaded from .env


In [63]:
paths = sorted(CV_INPUT_DIR.glob("*.pdf")) + sorted(CV_INPUT_DIR.glob("*.docx")) + sorted(CV_INPUT_DIR.glob("*.doc"))
paths = [p for p in paths if p.suffix.lower() in (".pdf", ".docx", ".doc")]
if not paths:
    raise SystemExit(f"No PDF/DOCX files found in {CV_INPUT_DIR}")
print(f"Found {len(paths)} CV(s): {[p.name for p in paths]}")

Found 2 CV(s): ['deHaan_CV_25-26.pdf', "O'Reilly_CV_25-26.docx"]


## Extract lines and metadata (from extract_text.ipynb)

In [64]:
import re
from io import BytesIO

MIME = {".pdf": "application/pdf", ".docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document", ".doc": "application/msword"}

def _pdf_lines(doc):
    try:
        import pdfplumber
        with pdfplumber.open(BytesIO(doc)) as pdf:
            out = []
            for page in pdf.pages:
                text = page.extract_text(layout=True) or ""
                out.extend(text.splitlines())
            return out
    except ImportError:
        pass
    from pypdf import PdfReader
    reader = PdfReader(BytesIO(doc))
    out = []
    for page in reader.pages:
        try:
            text = page.extract_text(extraction_mode="layout") or ""
        except Exception:
            text = page.extract_text() or ""
        out.extend(text.splitlines())
    return out

def _docx_lines(doc):
    from docx import Document
    doc = Document(BytesIO(doc))
    out = []
    for p in doc.paragraphs:
        out.append(p.text)
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for p in cell.paragraphs:
                    out.append(p.text)
    return out

def extract_lines(doc, mime, merge=True):
    if mime == "application/pdf":
        lines = _pdf_lines(doc)
    elif mime in ("application/vnd.openxmlformats-officedocument.wordprocessingml.document", "application/msword"):
        lines = _docx_lines(doc)
    else:
        raise ValueError(f"Unsupported: {mime}")
    return lines

In [65]:
def extract_line_metadata(document, mime, lines):
    from io import BytesIO
    numbered = [(i, ln.strip()) for i, ln in enumerate(lines, 1) if ln.strip()]
    if not numbered:
        return []
    def _format_hints():
        if mime == "application/pdf":
            return _pdf_format_hints(document, lines)
        if mime in ("application/vnd.openxmlformats-officedocument.wordprocessingml.document", "application/msword"):
            return _docx_format_hints(document, lines)
        return {}
    def _is_bold_font(fn):
        return fn and ("bold" in str(fn).lower() or "-b" in str(fn).lower() or "_b" in str(fn).lower())
    def _pdf_format_hints(doc, lns):
        try:
            import pdfplumber
        except ImportError:
            return {}
        plumber_lines, sizes = [], []
        with pdfplumber.open(BytesIO(doc)) as pdf:
            for page in pdf.pages:
                words = page.extract_words(extra_attrs=["size", "fontname"])
                if not words: continue
                rows = {}
                for w in words:
                    top = int(w.get("top", 0) or 0)
                    rows.setdefault(top, []).append(w)
                for top in sorted(rows):
                    ws = rows[top]
                    ws_sorted = sorted(ws, key=lambda w: (w.get("x0", 0), w.get("x1", 0)))
                    text = " ".join(w.get("text", "") or "" for w in ws_sorted).strip()
                    if not text: continue
                    szs = [float(w.get("size", 0) or 0) for w in ws if w.get("size") is not None]
                    max_sz = max(szs) if szs else 0
                    all_bold = bool(ws_sorted and all(_is_bold_font(w.get("fontname")) for w in ws_sorted))
                    if max_sz > 0: sizes.append(max_sz)
                    plumber_lines.append((text, max_sz, all_bold))
        if not sizes: return {}
        thresh = sorted(sizes)[len(sizes) // 2] * 1.1
        used = [False] * len(plumber_lines)
        numd = [(i, ln.strip()) for i, ln in enumerate(lns, 1) if ln.strip()]
        import unicodedata
        def norm(s): return unicodedata.normalize("NFKC", " ".join(s.split()))
        hints = {}
        for num, ln in numd:
            ln_n = norm(ln)
            for j, row in enumerate(plumber_lines):
                if not used[j] and norm(row[0]) == ln_n:
                    sz, bold = row[1], row[2]
                    hints[num] = (sz >= thresh and sz > 0) or bold
                    used[j] = True
                    break
            else:
                hints[num] = False
        return hints
    def _docx_format_hints(doc, lns):
        from docx import Document
        doc = Document(BytesIO(doc))
        hints, n = {}, 1
        for p in doc.paragraphs:
            style = (p.style and p.style.name or "").lower()
            runs = p.runs
            all_bold = bool(runs and all(getattr(r.font, "bold", None) for r in runs))
            hints[n] = "heading" in style or (all_bold and len(p.text.strip()) < 80)
            n += 1
        for t in doc.tables:
            for row in t.rows:
                for cell in row.cells:
                    for p in cell.paragraphs:
                        style = (p.style and p.style.name or "").lower()
                        runs = p.runs
                        all_bold = bool(runs and all(getattr(r.font, "bold", None) for r in runs))
                        hints[n] = "heading" in style or (all_bold and len(p.text.strip()) < 80)
                        n += 1
        return hints
    def _stem_line(line):
        from nltk.stem import SnowballStemmer
        stemmer = SnowballStemmer("english")
        words = re.findall(r"\b\w+\b", line.lower())
        return set(stemmer.stem(w) for w in words)
    def _categorize_header(line, kw_stems):
        line_stems = _stem_line(line)
        stemmed_str = " ".join(sorted(line_stems))
        best_cat, best_score = "other", 0
        for cat in ("publication", "presentation", "recognition"):
            score = len(line_stems & kw_stems[cat])
            if score > best_score: best_score, best_cat = score, cat
        return (best_cat if best_score > 0 else "other", stemmed_str)
    from nltk.stem import SnowballStemmer
    stemmer = SnowballStemmer("english")
    kw = {"publication": {"publication", "publications", "journal", "paper", "papers", "published", "article", "articles", "book", "books", "chapter", "chapters", "review", "working", "submitted", "accepted", "jae", "jar", "jf", "accounting", "round", "forthcoming", "revise", "revision", "conditionally"},
          "presentation": {"conference", "conferences", "workshop", "workshops", "keynote", "keynotes", "talk", "talks", "presentation", "presentations", "panel", "panels", "media", "interview", "interviews", "consortium", "faculty", "invited", "webinar", "podcast", "symposium"},
          "recognition": {"mentions", "mentioned", "recognition", "recognitions", "award", "awards", "honor", "honors", "fellowship", "fellowships", "grant", "grants", "prize", "prizes", "fellow", "professor", "chair", "selected", "distinguished", "outstanding", "accomplishment", "accomplishments", "appointment", "elected", "nominated"}}
    kw_stems = {cat: {stemmer.stem(w) for w in words} for cat, words in kw.items()}
    format_hints = _format_hints()
    out = []
    for num, ln in numbered:
        fmt = format_hints.get(num, False)
        ml_cat, stemmed = _categorize_header(ln, kw_stems)
        out.append({"line_number": num, "line": ln, "is_header_candidate": fmt, "source": ["format"] if fmt else ["none"], "ml_category": ml_cat if fmt else "", "stemmed_words": stemmed})
    out = sorted(out, key=lambda x: x["line_number"])
    headers = [(m["line_number"], m["ml_category"]) for m in out if m["is_header_candidate"] and m["ml_category"]]
    for m in out:
        if not m["is_header_candidate"]:
            n = m["line_number"]
            prev = [(h[0], h[1]) for h in headers if h[0] < n]
            nearest = sorted(prev, key=lambda h: n - h[0])[:1]
            m["nearest_ml_category"] = nearest[0][1] if nearest else ""
    return out

In [66]:
import json
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

def _clean_metadata(line_metadata):
    """Strip source/stemmed_words; keep all lines (no category filter)."""
    def _clean_obj(m):
        return {k: v for k, v in m.items() if k not in ("source", "stemmed_words")}
    return [_clean_obj(m) for m in line_metadata]

def _repair_truncated_json(s):
    idx = s.rfind("},")
    if idx < 0: idx = s.rfind("}")
    if idx >= 0:
        try:
            fixed = s[:idx + 1].rstrip().rstrip(",") + "]"
            return json.loads(fixed)
        except json.JSONDecodeError: pass
    return None

def parse_augmented(raw):
    t = raw.strip()
    if not t: raise RuntimeError("Empty LLM response")
    if "```json" in t:
        m = re.search(r"```json\s*([\s\S]*)", t)
        t = (m.group(1) or "").strip()
        if t.endswith("```"): t = t[:-3].strip()
    elif "```" in t:
        m = re.search(r"```\s*([\s\S]*?)```", t, re.DOTALL)
        t = m.group(1).strip() if m else t
    if t and not t.lstrip().startswith("["):
        m = re.search(r"\[[\s\S]*\]", t)
        t = m.group(0) if m else t
    try:
        augmented = json.loads(t)
    except json.JSONDecodeError as e:
        if "Unterminated" in str(e):
            augmented = _repair_truncated_json(t)
            if augmented is None: raise
        else: raise
    if isinstance(augmented, dict) and "lines" in augmented:
        augmented = augmented["lines"]
    if not isinstance(augmented, list):
        raise RuntimeError(f"Expected JSON array, got {type(augmented)}")
    return augmented

CSV_HEADERS = ["filename", "asset_type", "year", "title", "asset_sub_type", "status", "role", "institution"]

JOURNAL_MAP = {"JAE": "Journal of Accounting & Economics", "JAR": "Journal of Accounting Research", "JF": "Journal of Finance", "CAR": "Contemporary Accounting Research", "TAR": "The Accounting Review"}

def _fill_institution_from_line(institution, line):
    """When institution empty, infer from line using journal patterns."""
    if institution and str(institution).strip(): return institution
    line_upper = (line or "").upper()
    for abbr, full in JOURNAL_MAP.items():
        if abbr in line_upper or f" {abbr} " in f" {line_upper} " or line_upper.endswith(abbr):
            return full
    return institution or ""

def _proposed_to_row(m, p, filename):
    cat = m.get("ml_category") or m.get("nearest_ml_category") or "publication"
    inst = p.get("institution") or ""
    inst = _fill_institution_from_line(inst, m.get("line", ""))
    y = p.get("year")
    return {"filename": filename, "asset_type": cat, "year": "" if y is None else str(y), "title": p.get("title") or "", "asset_sub_type": p.get("type") or "", "status": p.get("status") or "", "role": p.get("role") or "", "institution": inst}

def _deduplicate_rows(rows):
    """Merge rows with same filename+title+year; prefer published over in_progress."""
    by_key = {}
    for r in rows:
        k = (r["filename"], (r["title"] or "").strip(), str(r.get("year", "")).strip())
        existing = by_key.get(k)
        if not existing or (r.get("status") == "published" and existing.get("status") != "published"):
            by_key[k] = r
    return list(by_key.values())

In [67]:
def process_one_cv(path, on_progress=None):
    """Process single CV: extract, metadata, clean, LLM augment. Returns list of CSV rows.
    on_progress(tokens, est) called once when LLM response received."""
    suffix = path.suffix.lower()
    mime = MIME.get(suffix)
    if not mime: return []
    document = path.read_bytes()
    lines = extract_lines(document, mime, merge=False)
    line_metadata = extract_line_metadata(document, mime, lines)
    cleaned = _clean_metadata(line_metadata)
    if not cleaned: return []
    TEMP_DIR.mkdir(parents=True, exist_ok=True)
    (TEMP_DIR / f"{path.stem}_line_metadata.json").write_text(json.dumps(line_metadata, indent=2), encoding="utf-8")
    (TEMP_DIR / f"{path.stem}_line_metadata_cleaned.json").write_text(json.dumps(cleaned, indent=2), encoding="utf-8")
    metadata_json = json.dumps(cleaned, indent=2)
    est_output = 2 * (len(metadata_json) // 4)
    prompt = f"""AUGMENT the line metadata below. Each object has line_number, line, is_header_candidate, ml_category, nearest_ml_category.

CRITICAL: Do NOT drop any lines. Output array MUST have exactly {len(cleaned)} objects, same order, same line_number values.

1) CATEGORIZATION: Correct ml_category or nearest_ml_category if wrong.

2) MERGE LINES (only when needed): Only merge when lines were incorrectly broken up. When merging: update "line" on first line; for continuation lines add "merged_into": line_number and "proposed": null. Keep all objects.

3) PROPOSED OBJECT: For each publication, presentation, or recognition item add "proposed" with the pydantic-shaped object. Use null for headers, pii, continuation lines.

Proposed shapes:
- publication: {{"year": 2020, "type": "book|journal|other", "status": "in_progress|published", "institution": "", "title": "", "role": "sole_author|co_author"}}
- presentation: {{"title": "", "year": 2020, "type": "conference|keynote|media|workshop|other", "role": "sole_presenter|co_presenter", "institution": ""}}
- recognition: {{"year": 2020, "title": "", "institution": ""}}

Rules: institution = publisher for books, journal name for articles. role sole_author when CV owner only author, co_author when multiple or "with X". When author info is missing (e.g. working paper titles only), default to co_author.

Recognition: Include service roles (Managing Editor, Reviewer, Director, Editor, Faculty Member, Committee member) as recognition items with proposed objects. Use title for the role description, institution for the organization/journal.

REFERENCE - Journal abbreviations:
JAE = Journal of Accounting & Economics | JAR = Journal of Accounting Research | JF = Journal of Finance | CAR = Contemporary Accounting Research | TAR = The Accounting Review
Infer institution from line text: "Under review at JAE" → institution = "Journal of Accounting & Economics"; "Submitted to JF" → "Journal of Finance".

Years: "2024-25" → 2025; "this year"/"current" → use current year (2026); "forthcoming"/"in press" → null or most recent.

Example (publication):
Input: {{"line_number": 10, "line": "Smith, J. (2023). Title here. Journal of Accounting & Economics, 47(2), 123-145.", "ml_category": "publication"}}
Output: add "proposed": {{"year": 2023, "type": "journal", "status": "published", "institution": "Journal of Accounting & Economics", "title": "Title here", "role": "sole_author"}}

---
LINE METADATA:
{metadata_json}"""
    client = OpenAI()
    response = client.chat.completions.create(
        model=OPENAI_MODEL or os.environ.get("OPENAI_MODEL", "gpt-5.2"),
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
    )
    raw = response.choices[0].message.content or ""
    if on_progress:
        on_progress(len(raw) // 4, est_output)
    augmented = parse_augmented(raw)
    aug_path = TEMP_DIR / f"{path.stem}_line_metadata_augmented.json"
    aug_path.write_text(json.dumps(augmented, indent=2), encoding="utf-8")
    rows = [_proposed_to_row(m, m["proposed"], path.name) for m in augmented if m.get("proposed")]
    return rows

In [68]:
import csv
PROGRESS_FILE = TEMP_DIR / "batch_progress.txt"
TEMP_DIR.mkdir(parents=True, exist_ok=True)
progress_data = {}
progress_lock = threading.Lock()

def write_progress():
    with progress_lock:
        lines = []
        for name, d in sorted(progress_data.items()):
            s, e = d.get("streamed", 0), d.get("est", 0)
            pct = min(99, 100 * s // e) if e else 0
            status = "done" if d.get("done") else f"{pct}%"
            lines.append(f"{name}: {status} (~{s} tokens)")
        PROGRESS_FILE.write_text("\n".join(lines) if lines else "Starting...")

all_rows = []
n = len(paths)
print(f"Starting batch: {n} file(s). Live progress: tail -f {PROGRESS_FILE}")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {}
    for path in paths:
        progress_data[path.name] = {"streamed": 0, "est": 0, "done": False}
        def make_cb(p):
            def cb(streamed, est):
                with progress_lock:
                    progress_data[p.name]["streamed"] = streamed
                    progress_data[p.name]["est"] = est
                write_progress()
            return cb
        futures[ex.submit(process_one_cv, path, on_progress=make_cb(path))] = path
    write_progress()
    for future in as_completed(futures):
        path = futures[future]
        with progress_lock:
            progress_data[path.name]["done"] = True
        try:
            rows = future.result()
            csv_path = TEMP_DIR / f"{path.stem}_output.csv"
            json_path = TEMP_DIR / f"{path.stem}_output.json"
            with csv_path.open("w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=CSV_HEADERS)
                w.writeheader()
                w.writerows(rows)
            json_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
            all_rows.extend(rows)
        except Exception as e:
            pass  # progress already updated
        write_progress()
PROGRESS_FILE.write_text("Complete.")
print(f"Total: {len(all_rows)} rows from {len(paths)} file(s)")

Starting batch: 2 file(s). Live progress: tail -f /Users/albert.wong/code/github/albertcwong/cv-parser2/tmp/batch_progress.txt
Total: 163 rows from 2 file(s)


In [69]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
rows_to_write = _deduplicate_rows(all_rows) if DEDUPLICATE_ROWS else all_rows
if OUTPUT_FORMAT == "csv":
    import csv
    with OUTPUT_FILE.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CSV_HEADERS)
        w.writeheader()
        w.writerows(rows_to_write)
    print(f"Wrote {len(rows_to_write)} rows to {OUTPUT_FILE}")
else:
    OUTPUT_FILE.write_text(json.dumps(rows_to_write, indent=2), encoding="utf-8")
    print(f"Wrote {len(rows_to_write)} rows to {OUTPUT_FILE}")

Wrote 160 rows to /Users/albert.wong/code/github/albertcwong/cv-parser2/output/combined.csv
